# Notebook 2 — Zero-shot Llama 3 8B (base and Instruct) on SciEntsBank  ·  GPU: T4

Classifies SciEntsBank answers with the untuned models, using the SciEntsBank prompt that fine-tuning will use. The prediction is read from the next-token probabilities of the five label words (one forward pass, no sampling). 3-way and 2-way predictions are derived from the 5-way probabilities.

Two versions are reported: **raw** (argmax of the model's label probabilities) and **calibrated** (logistic regression on the five label log-probabilities, fitted on train+val only), the counterpart of the rescaled Mohler scores.

The three test sets are scored first, then train+val (needed only for calibration). **Before running:** upload the updated `llm_utils.py` to `MyDrive/asag_revision/`. T4 runtime. Resumable. Expected time: ~60–75 min (~10,800 answers × 2 models).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install -U bitsandbytes accelerate peft

In [ ]:
import json, gc, platform
import numpy as np, pandas as pd, torch, transformers, bitsandbytes
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from asag_utils import *
from llm_utils import *

versions = {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__,
            'bitsandbytes': bitsandbytes.__version__, 'gpu': torch.cuda.get_device_name(0)}
se = pd.read_csv(f'{ROOT}/data/scientsbank_prepared.csv')
prompts = make_prompts(se, 'scientsbank')
PRIORITY = ['test_ua', 'test_uq', 'test_ud', 'val', 'train']
se.split.value_counts().reindex(PRIORITY)

## 1. Score every answer with each model (tests first; resumable, saves every 256 answers)

In [ ]:
MODELS = {'llama3_base_zs': (BASE_MODEL, False), 'llama3_instruct_zs': (INSTRUCT_MODEL, True)}
CHUNK = 256
lp_cols = [f'lp_{l}' for l in LABEL_ORDER]

for tag, (name, chat) in MODELS.items():
    path = f'{ROOT}/results/scientsbank_{tag}.csv'
    done_ids = set(pd.read_csv(path).uid) if os.path.exists(path) else set()
    if len(done_ids) >= len(se):
        print(tag, 'already complete'); continue
    model, tok = load_hf_4bit(name)
    words = label_candidates(chat)
    cand, full_ids = candidate_ids(tok, words)
    print(dict(zip(words, full_ids)))
    texts = to_chat(tok, prompts, INSTRUCT_PREFILL['scientsbank']) if chat else prompts
    add_special = not chat

    # sanity check: each label's first token is exactly what follows the prompt when the label is appended
    p_ids = tok(texts[0], add_special_tokens=add_special)['input_ids']
    for w, c in zip(words, cand):
        full = tok(texts[0] + w, add_special_tokens=add_special)['input_ids']
        assert full[:len(p_ids)] == p_ids and full[len(p_ids)] == c, f'tokenisation mismatch for {w!r}'

    order = se.assign(pr=se.split.map(PRIORITY.index)).sort_values('pr', kind='stable').index
    todo = [i for i in order if se.uid[i] not in done_ids]
    for start in tqdm(range(0, len(todo), CHUNK), desc=tag):
        idx = todo[start:start + CHUNK]
        lp = candidate_logprobs(model, tok, [texts[i] for i in idx], cand, batch_size=8, add_special_tokens=add_special)
        chunk = pd.concat([pd.DataFrame({'uid': se.uid.values[idx]}), pd.DataFrame(lp, columns=lp_cols)], axis=1)
        chunk.to_csv(path, mode='a', header=not os.path.exists(path), index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

## 2. Raw and calibrated predictions for each scheme

In [ ]:
preds = pd.read_csv(f'{ROOT}/results/scientsbank_preds.csv')
label_col = {'5way': 'label', '3way': 'label3', '2way': 'label2'}
diag = []
for tag in MODELS:
    z = pd.read_csv(f'{ROOT}/results/scientsbank_{tag}.csv').drop_duplicates('uid')
    d = se[['uid', 'split'] + list(label_col.values())].merge(z, on='uid')
    q, mass, raw = labels_from_logprobs(d[lp_cols].values)
    new = pd.DataFrame({'uid': d.uid})
    for scheme in label_col:
        new[f'{tag}_{scheme}'] = raw[scheme]
    # calibration: logistic regression on the 5 renormalised log-probs, fitted on train+val only
    feats = np.log(np.clip(q, 1e-12, 1))
    fit = d.split.isin(['train', 'val']).values
    if fit.sum() == (se.split.isin(['train', 'val'])).sum():
        for scheme, col in label_col.items():
            clf = LogisticRegression(max_iter=2000).fit(feats[fit], d[col].values[fit])
            new[f'{tag}_cal_{scheme}'] = clf.predict(feats)
    else:
        print(f'{tag}: train/val not fully scored yet, calibrated version skipped')
    preds = preds.drop(columns=[c for c in new.columns if c != 'uid' and c in preds.columns]).merge(new, on='uid', how='left')
    t = d.split.str.startswith('test').values
    diag.append({'model': tag, 'scored': len(d), 'mean_valid_mass': mass.mean(), 'min_valid_mass': mass.min(),
                 **{f'pred_{l}': (raw['5way'][t] == l).mean() for l in LABEL_ORDER}})
preds.to_csv(f'{ROOT}/results/scientsbank_preds.csv', index=False)
gold = se[se.split.str.startswith('test')].label.value_counts(normalize=True)
display(pd.DataFrame(diag).round(3))
print('Gold 5-way distribution on the test sets:', gold.round(3).to_dict())

`valid_mass` = probability the model puts on the five label words; `pred_*` = share of test answers given each label (compare with the gold distribution to spot a bias towards one label).

In [ ]:
TESTS = ['test_ua', 'test_uq', 'test_ud']
systems = [c.rsplit('_', 1)[0] for c in preds.columns if c.startswith('llama3_') and c.endswith('_5way')]
rows = []
for scheme, col in label_col.items():
    for name in systems:
        for t in TESTS:
            d = preds[(preds.split == t) & preds[f'{name}_{scheme}'].notna()]
            if len(d) < (preds.split == t).sum():
                print(f'WARNING: {name} scored only {len(d)} answers of {t}')
            groups = None if t == 'test_ua' else d.group
            rows.append({'scheme': scheme, 'system': name, 'test': t, 'n': len(d),
                         **summarise(d[col], d[f'{name}_{scheme}'], cls_metrics, groups=groups)})
nb2 = pd.DataFrame(rows)
nb2.to_csv(f'{ROOT}/results/scientsbank_nb2_summary.csv', index=False)
ref = pd.read_csv(f'{ROOT}/results/scientsbank_nb0_summary.csv')
both = pd.concat([ref[ref.system.isin(['majority', 'lexical_4'])], nb2])
for metric in ['macro_f1', 'acc']:
    print(metric); display(both.pivot_table(index=['scheme', 'system'], columns='test', values=metric).round(3))

In [ ]:
versions['peft'] = __import__('peft').__version__
json.dump(versions, open(f'{ROOT}/results/env_nb2.json', 'w'), indent=2)
versions